In [1]:
import opt_einsum as oe
import numpy as np
import torch
import sys
sys.path.append("../../")
import mps
from mps.trainer.data_utils import SyntheticDataset, SyntheticDatasetV2

In [2]:
N = 256
dataset = SyntheticDatasetV2(n=N, num_samples=(2**15), seed=42)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2**10, shuffle=True)

In [3]:
from mps import tpcp_mps
from mps.trainer.utils import calculate_accuracy, loss_batch
# --- Step 2: Build and Prepare TPCP ---
tpcp = tpcp_mps.MPSTPCP(
    N,
    K=8,
    d=2,
    enable_r=False,
    with_identity=False,
    manifold=tpcp_mps.ManifoldType.EXACT,
)
tpcp.train()

logsoftmax = torch.nn.LogSoftmax(dim=-1)
nnloss = torch.nn.NLLLoss(reduction="mean")

data_batch, target_batch = next(iter(dataloader))
# Initialize W: start with first column ones and second column small.

In [29]:
i = 2
x = data_batch[i]

rho0 = torch.einsum("i, j->ij", x[0], x[0].conj())
rho1 = torch.einsum("i, j->ij", x[1], x[1].conj())
rho_in = torch.einsum("ij,kl->ikjl", rho0, rho1).reshape(4, 4)
K = tpcp.kraus_ops[0].data.reshape(8, 4, 4)
rho_k_out = torch.einsum("kij, jl, kml->im", K, rho_in, K.conj()).reshape(2, 2, 2, 2)
#partial trace
rho_k_out = torch.einsum("ijil -> jl", rho_k_out)

for k in range(1, len(tpcp.kraus_ops.kraus_ops)):
    K = tpcp.kraus_ops[k].data.reshape(8, 4, 4)
    rho_new = torch.einsum("i, j->ij", x[k+1], x[k+1].conj())
    rho_k_in = torch.einsum("ij,kl->ikjl", rho_k_out, rho_new).reshape(4, 4)
    rho_k_out = torch.einsum("kij, jl, kml->im", K, rho_k_in, K.conj()).reshape(2, 2, 2, 2)
    rho_last = rho_k_out.data.reshape(4, 4)
    #partial trace
    rho_k_out = torch.einsum("ijil -> jl", rho_k_out)

mes0 = tpcp.r.conj() @ tpcp.pros0 @ tpcp.r.T - tpcp.r.conj() @ tpcp.pros1 @ tpcp.r.T

print(torch.trace(mes0 @ rho_k_out), target_batch[i][1])
print(tpcp(data_batch[i].unsqueeze(0)), target_batch[i])

IndexError: index 1 is out of bounds for dimension 0 with size 0

In [24]:
rho_last

tensor([[ 0.3219, -0.0172,  0.0211, -0.0824],
        [-0.0172,  0.1968, -0.0304,  0.0495],
        [ 0.0211, -0.0304,  0.3291,  0.1109],
        [-0.0824,  0.0495,  0.1109,  0.1523]], dtype=torch.float64)

In [25]:
tpcp.rho_list[-1]

tensor([[[ 0.3219, -0.0172,  0.0211, -0.0824],
         [-0.0172,  0.1968, -0.0304,  0.0495],
         [ 0.0211, -0.0304,  0.3291,  0.1109],
         [-0.0824,  0.0495,  0.1109,  0.1523]]], dtype=torch.float64,
       grad_fn=<SumBackward1>)